In [1]:
import joblib

scaler = joblib.load(
    "cross_device_feature_scaler.pkl"
)

print(type(scaler))
print(scaler.mean_)
print(scaler.scale_)

<class 'sklearn.preprocessing._data.StandardScaler'>
[ 54.30486526  69.35910699 186.81479667 277.07422832  34.97542381
  87.1288927 ]
[33.47473048  3.97927094 71.08608066 10.80642228 14.65841708 11.04398318]


/media/rehnoor/48984F97984F8284/Capstone Project LSTM Branch/Prediction Service/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
import os
import zipfile
import tensorflow as tf

MODEL_PATH = "best_cross_device_lstm.keras"

print("File exists:", os.path.exists(MODEL_PATH))
print("File size:", os.path.getsize(MODEL_PATH), "bytes")
print("Is ZIP archive:", zipfile.is_zipfile(MODEL_PATH))

print("\nContents:")
with zipfile.ZipFile(MODEL_PATH, "r") as z:
    for name in z.namelist():
        print(" -", name)

I0000 00:00:1783480820.661471   31447 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783480821.018681   31447 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783480829.800490   31447 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


File exists: True
File size: 129863 bytes
Is ZIP archive: True

Contents:
 - metadata.json
 - config.json
 - model.weights.h5


In [3]:
import tensorflow as tf

MODEL_PATH = "best_cross_device_lstm.keras"

print("TensorFlow version:", tf.__version__)
print("Loading model...")

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("\nModel loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

model.summary()

TensorFlow version: 2.21.0
Loading model...

Model loaded successfully!
Input shape: (None, 20, 6)
Output shape: (None, 1)


E0000 00:00:1783480850.615324   31447 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 40)             │         7,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,193 (32.00 KB)

 Trainable params: 8,193 (32.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import joblib
import tensorflow as tf

MODEL_PATH = "best_cross_device_lstm.keras"
SCALER_PATH = "cross_device_feature_scaler.pkl"

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

scaler = joblib.load(SCALER_PATH)

# Create 20 timesteps with 6 features.
# Using the scaler means as realistic neutral test values.
test_sequence = np.tile(
    scaler.mean_,
    (20, 1)
)

print("Original sequence shape:", test_sequence.shape)

# Scale exactly as during training
scaled_sequence = scaler.transform(test_sequence)

# Add batch dimension
model_input = scaled_sequence.reshape(1, 20, 6)

print("Model input shape:", model_input.shape)

# Run inference
prediction = model.predict(
    model_input,
    verbose=0
)

print("\nPrediction:", prediction)
print("Predicted Delta T:", float(prediction[0][0]))

/media/rehnoor/48984F97984F8284/Learning LSTM/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/media/rehnoor/48984F97984F8284/Learning LSTM/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Original sequence shape: (20, 6)
Model input shape: (1, 20, 6)

Prediction: [[1.3356022]]
Predicted Delta T: 1.3356021642684937


In [7]:
import numpy as np
import tensorflow as tf
import tf2onnx
import onnxruntime as ort


KERAS_MODEL_PATH = "best_cross_device_lstm.keras"
ONNX_MODEL_PATH = "cross_device_lstm.onnx"


# ---------------------------------------------------------
# 1. LOAD KERAS MODEL
# ---------------------------------------------------------

print("\nLoading Keras model...")

keras_model = tf.keras.models.load_model(
    KERAS_MODEL_PATH,
    compile=False
)

print("Keras model loaded successfully.")
print("Input shape:", keras_model.input_shape)
print("Output shape:", keras_model.output_shape)


# ---------------------------------------------------------
# 2. CREATE EXPLICIT INFERENCE FUNCTION
# ---------------------------------------------------------

input_signature = [
    tf.TensorSpec(
        shape=(None, 20, 6),
        dtype=tf.float32,
        name="telemetry_input"
    )
]


@tf.function(
    input_signature=input_signature
)
def inference_function(telemetry_input):

    prediction = keras_model(
        telemetry_input,
        training=False
    )

    return {
        "predicted_delta_t": prediction
    }


# ---------------------------------------------------------
# 3. CONVERT FUNCTION TO ONNX
# ---------------------------------------------------------

print("\nConverting model to ONNX...")

tf2onnx.convert.from_function(
    inference_function,
    input_signature=input_signature,
    opset=17,
    output_path=ONNX_MODEL_PATH
)

print("ONNX model saved successfully.")
print("Saved at:", ONNX_MODEL_PATH)


# ---------------------------------------------------------
# 4. CREATE IDENTICAL TEST INPUT
# ---------------------------------------------------------

np.random.seed(42)

test_input = np.random.randn(
    1,
    20,
    6
).astype(np.float32)

print("\nTest input shape:", test_input.shape)


# ---------------------------------------------------------
# 5. KERAS PREDICTION
# ---------------------------------------------------------

keras_prediction = keras_model.predict(
    test_input,
    verbose=0
)

keras_delta_t = float(
    keras_prediction[0][0]
)


# ---------------------------------------------------------
# 6. ONNX PREDICTION
# ---------------------------------------------------------

onnx_session = ort.InferenceSession(
    ONNX_MODEL_PATH,
    providers=["CPUExecutionProvider"]
)

onnx_input = onnx_session.get_inputs()[0]
onnx_output = onnx_session.get_outputs()[0]

print("\nONNX input:")
print("Name :", onnx_input.name)
print("Shape:", onnx_input.shape)
print("Type :", onnx_input.type)

print("\nONNX output:")
print("Name :", onnx_output.name)
print("Shape:", onnx_output.shape)
print("Type :", onnx_output.type)


onnx_prediction = onnx_session.run(
    None,
    {
        onnx_input.name: test_input
    }
)


onnx_delta_t = float(
    np.asarray(onnx_prediction[0]).reshape(-1)[0]
)


# ---------------------------------------------------------
# 7. COMPARE
# ---------------------------------------------------------

difference = abs(
    keras_delta_t - onnx_delta_t
)

print("\n========================================")
print("       CONVERSION VALIDATION RESULT")
print("========================================")

print(f"Keras Delta T : {keras_delta_t:.10f}")
print(f"ONNX Delta T  : {onnx_delta_t:.10f}")
print(f"Difference    : {difference:.10f}")


if difference < 1e-4:

    print(
        "\nSUCCESS: ONNX model matches Keras model."
    )

else:

    print(
        "\nWARNING: Predictions differ more than expected."
    )


Loading Keras model...
Keras model loaded successfully.
Input shape: (None, 20, 6)
Output shape: (None, 1)

Converting model to ONNX...


I0000 00:00:1783482859.055916   31447 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1783482859.271097   31447 single_machine.cc:376] Starting new session
I0000 00:00:1783482859.821266   31447 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1783482859.821389   31447 single_machine.cc:376] Starting new session
TF freezing failed. Attempting to fix freezing errors.
Removed SelectV2 sequential_1/lstm_1/AssignVariableOp_3
Removed GreaterEqual sequential_1/lstm_1/AssignVariableOp_2
Removed StatelessRandomUniformV2 sequential_1/lstm_1/AssignVariableOp_1
Removed MatMul sequential_1/lstm_1/AssignVariableOp
I0000 00:00:1783482859.927929   31447 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1783482859.928074   31447 single_machine.cc:376] Starting new session
I0000 00:00:1783482860.203951   31447 mlir_graph_optimization_pass.cc:437] MLIR V1 o

ONNX model saved successfully.
Saved at: cross_device_lstm.onnx

Test input shape: (1, 20, 6)

ONNX input:
Name : telemetry_input
Shape: ['unk__260', 20, 6]
Type : tensor(float)

ONNX output:
Name : predicted_delta_t
Shape: ['unk__261', 1]
Type : tensor(float)

       CONVERSION VALIDATION RESULT
Keras Delta T : -2.3805727959
ONNX Delta T  : -2.3805725574
Difference    : 0.0000002384

SUCCESS: ONNX model matches Keras model.
